# CXR Triage — Clean Evaluation Notebook

This notebook evaluates a single trained model from scratch with built-in
sanity checks at every step, before computing final test-set metrics.

**Model under evaluation:** DenseNet-121 + Focal Loss + CLAHE + 320x320
(corrected logits pipeline)

**Methodology:** thresholds are derived from the VALIDATION set only, then
frozen and applied to the TEST set. No test-set information is used to
pick thresholds, avoiding data leakage.

## 1. Setup and imports

In [1]:
import sys
sys.path.append('D:/cxr-triage')

import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    precision_recall_curve
)
from tqdm import tqdm

from src.data.dataset import ChestXrayDataset
from src.data.transforms import get_val_transforms
from src.models.densenet import DenseNetModel

LABELS = [
    'Atelectasis', 'Consolidation', 'Infiltration',
    'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis',
    'Effusion', 'Pneumonia', 'Pleural_Thickening',
    'Cardiomegaly', 'Nodule', 'Mass', 'Hernia'
]
CRITICAL = ['Pneumothorax', 'Edema', 'Pneumonia', 'Hernia']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGE_ROOT = "F:/X ray dataset/Second Version"

CHECKPOINT_PATH = "D:/cxr-triage/checkpoints/clahe_320_logits_fix/best_model.pth"
IMAGE_SIZE = 320
USE_CLAHE = True

print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Image size: {IMAGE_SIZE}, CLAHE: {USE_CLAHE}")

Device: cuda
Checkpoint: D:/cxr-triage/checkpoints/clahe_320_logits_fix/best_model.pth
Image size: 320, CLAHE: True


## 2. Sanity check — data splits exist and have zero patient leakage

In [2]:
train_df = pd.read_csv('D:/cxr-triage/data/processed/train.csv')
val_df = pd.read_csv('D:/cxr-triage/data/processed/val.csv')
test_df = pd.read_csv('D:/cxr-triage/data/processed/test.csv')

print(f"Train: {len(train_df)} images")
print(f"Val:   {len(val_df)} images")
print(f"Test:  {len(test_df)} images")

train_p = set(train_df['Patient ID'])
val_p = set(val_df['Patient ID'])
test_p = set(test_df['Patient ID'])

tv = len(train_p & val_p)
tt = len(train_p & test_p)
vt = len(val_p & test_p)

print(f"\nTrain-Val patient overlap:  {tv}")
print(f"Train-Test patient overlap: {tt}")
print(f"Val-Test patient overlap:   {vt}")

assert tv == 0 and tt == 0 and vt == 0, "LEAKAGE DETECTED — STOP"
print("\nPASS: zero patient leakage across all splits")

Train: 68918 images
Val:   17606 images
Test:  25596 images

Train-Val patient overlap:  0
Train-Test patient overlap: 0
Val-Test patient overlap:   0

PASS: zero patient leakage across all splits


## 3. Load model and sanity check the checkpoint

In [3]:
model = DenseNetModel(num_classes=14, pretrained=False).to(DEVICE)
checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Checkpoint epoch: {checkpoint['epoch']+1}")
print(f"Checkpoint train-time best AUC: {checkpoint['best_auc']:.4f}")

# Sanity check: confirm classifier has no Sigmoid (raw logits architecture)
classifier = model.model.classifier
print(f"\nClassifier module: {classifier}")
is_linear_only = isinstance(classifier, torch.nn.Linear)
print(f"PASS: classifier is plain Linear (no Sigmoid)" if is_linear_only
      else "FAIL: classifier is not plain Linear — check architecture")

Checkpoint epoch: 15
Checkpoint train-time best AUC: 0.7943

Classifier module: Linear(in_features=1024, out_features=14, bias=True)
PASS: classifier is plain Linear (no Sigmoid)


## 4. Sanity check — one forward pass produces unbounded logits

In [4]:
sanity_transform = get_val_transforms(image_size=IMAGE_SIZE, use_clahe=USE_CLAHE)
sanity_dataset = ChestXrayDataset(csv_path=None, image_root=IMAGE_ROOT, transform=sanity_transform)
sanity_dataset.df = val_df.head(16).reset_index(drop=True)
sanity_loader = DataLoader(sanity_dataset, batch_size=16, shuffle=False)

with torch.no_grad():
    sample_images, sample_targets = next(iter(sanity_loader))
    sample_images = sample_images.to(DEVICE)
    sample_logits = model(sample_images)

lo, hi = sample_logits.min().item(), sample_logits.max().item()
print(f"Logit min: {lo:.4f}")
print(f"Logit max: {hi:.4f}")
print(f"Contains NaN: {torch.isnan(sample_logits).any().item()}")
print(f"Contains Inf: {torch.isinf(sample_logits).any().item()}")

bounded = (0.0 <= lo) and (hi <= 1.0)
print("\nFAIL: outputs look bounded to [0,1] — sigmoid bug may be present" if bounded
      else "PASS: outputs are unbounded raw logits, as expected")

Logit min: -5.7109
Logit max: 3.3177
Contains NaN: False
Contains Inf: False
PASS: outputs are unbounded raw logits, as expected


## 5. Inference helper function

In [5]:
def get_predictions(df, batch_size=8):
    transform = get_val_transforms(image_size=IMAGE_SIZE, use_clahe=USE_CLAHE)
    dataset = ChestXrayDataset(csv_path=None, image_root=IMAGE_ROOT, transform=transform)
    dataset.df = df
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                         num_workers=4, pin_memory=True)

    all_logits, all_targets = [], []
    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Inference"):
            images = images.to(DEVICE)
            with torch.amp.autocast('cuda'):
                logits = model(images)
            all_logits.append(logits.cpu().numpy())
            all_targets.append(targets.numpy())

    logits = np.concatenate(all_logits, axis=0).astype(np.float32)
    targets = np.concatenate(all_targets, axis=0).astype(np.float32)
    probs = 1 / (1 + np.exp(-logits))
    return probs, targets

print("Inference function ready")

Inference function ready


## 6. Run inference on VALIDATION set

In [6]:
val_probs, val_targets = get_predictions(val_df)

print(f"Val probs shape: {val_probs.shape}")
print(f"Val targets shape: {val_targets.shape}")
print(f"Probs min/max: {val_probs.min():.4f} / {val_probs.max():.4f}")

in_range = (val_probs.min() >= 0.0) and (val_probs.max() <= 1.0)
print("PASS: probabilities correctly bounded [0,1]" if in_range
      else "FAIL: probabilities out of expected range")

Inference: 100%|██████████| 2201/2201 [03:35<00:00, 10.20it/s]

Val probs shape: (17606, 14)
Val targets shape: (17606, 14)
Probs min/max: 0.0000 / 0.9990
PASS: probabilities correctly bounded [0,1]


## 7. Find optimal per-class thresholds — VALIDATION SET ONLY
(this is the leak-free step — test set is never touched here)

In [7]:
optimal_thresholds = {}

print(f"{'Finding':<22} {'Threshold':<12} {'Val F1':<10}")
print("-" * 50)

for i, label in enumerate(LABELS):
    precisions, recalls, thresh = precision_recall_curve(val_targets[:, i], val_probs[:, i])
    f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
    best_idx = np.argmax(f1s)
    optimal_thresholds[label] = float(thresh[best_idx])
    print(f"{label:<22} {thresh[best_idx]:<12.4f} {f1s[best_idx]:<10.4f}")

print("-" * 50)
print("\nThresholds frozen. Will now be applied to TEST set only.")

Finding                Threshold    Val F1    
--------------------------------------------------
Atelectasis            0.7033       0.3386    
Consolidation          0.7491       0.1939    
Infiltration           0.6216       0.3657    
Pneumothorax           0.6756       0.2179    
Edema                  0.8820       0.2497    
Emphysema              0.7520       0.1932    
Fibrosis               0.8461       0.1506    
Effusion               0.7398       0.5045    
Pneumonia              0.7672       0.0690    
Pleural_Thickening     0.7443       0.1627    
Cardiomegaly           0.9158       0.3472    
Nodule                 0.6746       0.2091    
Mass                   0.7765       0.2998    
Hernia                 0.8972       0.4571    
--------------------------------------------------

Thresholds frozen. Will now be applied to TEST set only.


## 8. Run inference on TEST set (held-out, never used for any tuning)

In [8]:
test_probs, test_targets = get_predictions(test_df)

print(f"Test probs shape: {test_probs.shape}")
print(f"Test targets shape: {test_targets.shape}")

Inference: 100%|██████████| 3200/3200 [05:00<00:00, 10.64it/s]

Test probs shape: (25596, 14)
Test targets shape: (25596, 14)


## 9. Apply FROZEN validation thresholds to test set, compute final metrics

In [9]:
test_pred = np.zeros_like(test_probs, dtype=np.float32)
for i, label in enumerate(LABELS):
    test_pred[:, i] = (test_probs[:, i] >= optimal_thresholds[label]).astype(np.float32)

print(f"{'Finding':<22} {'Tier':<10} {'AUC':<8} {'F1':<8} {'Precision':<12} {'Recall':<8} {'Support':<8}")
print("-" * 85)

triage_map = {
    'Pneumothorax': 'Critical', 'Edema': 'Critical',
    'Pneumonia': 'Critical', 'Hernia': 'Critical',
    'Consolidation': 'Urgent', 'Effusion': 'Urgent',
    'Cardiomegaly': 'Urgent', 'Mass': 'Urgent',
    'Atelectasis': 'Urgent', 'Nodule': 'Urgent',
    'Infiltration': 'Urgent', 'Emphysema': 'Routine',
    'Fibrosis': 'Routine', 'Pleural_Thickening': 'Routine'
}

per_class_auc = []
for i, label in enumerate(LABELS):
    auc = roc_auc_score(test_targets[:, i], test_probs[:, i])
    per_class_auc.append(auc)
    f1 = f1_score(test_targets[:, i], test_pred[:, i], zero_division=0)
    prec = precision_score(test_targets[:, i], test_pred[:, i], zero_division=0)
    rec = recall_score(test_targets[:, i], test_pred[:, i], zero_division=0)
    support = int(test_targets[:, i].sum())
    tier = triage_map[label]
    print(f"{label:<22} {tier:<10} {auc:<8.4f} {f1:<8.4f} {prec:<12.4f} {rec:<8.4f} {support:<8}")

print("-" * 85)

mean_auc = np.mean(per_class_auc)
critical_auc = np.mean([auc for label, auc in zip(LABELS, per_class_auc) if label in CRITICAL])
micro_f1 = f1_score(test_targets, test_pred, average='micro')
macro_f1 = f1_score(test_targets, test_pred, average='macro')

print(f"\nMean Test AUC:     {mean_auc:.4f}")
print(f"Critical Test AUC: {critical_auc:.4f}")
print(f"Micro F1:          {micro_f1:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")

Finding                Tier       AUC      F1       Precision    Recall   Support 
-------------------------------------------------------------------------------------
Atelectasis            Urgent     0.7225   0.3406   0.2622       0.4858   3279    
Consolidation          Urgent     0.7210   0.2271   0.1483       0.4843   1815    
Infiltration           Urgent     0.6761   0.4437   0.3064       0.8037   6112    
Pneumothorax           Critical   0.8096   0.3868   0.2717       0.6705   2665    
Edema                  Critical   0.8172   0.1951   0.1510       0.2757   925     
Emphysema              Routine    0.7996   0.2625   0.1836       0.4602   1093    
Fibrosis               Routine    0.7643   0.0892   0.0729       0.1149   435     
Effusion               Urgent     0.7968   0.4863   0.4092       0.5992   4658    
Pneumonia              Critical   0.6570   0.0694   0.0515       0.1063   555     
Pleural_Thickening     Routine    0.7256   0.1680   0.1109       0.3465   1143    
C

## 10. Save results to disk

In [ ]:
import json

results = {
    'model': 'DenseNet-121 + Focal Loss + CLAHE + 320x320 (corrected pipeline)',
    'checkpoint': CHECKPOINT_PATH,
    'checkpoint_epoch': checkpoint['epoch'] + 1,
    'mean_test_auc': float(mean_auc),
    'critical_test_auc': float(critical_auc),
    'micro_f1': float(micro_f1),
    'macro_f1': float(macro_f1),
    'thresholds': optimal_thresholds,
    'per_class': {
        label: {
            'auc': float(roc_auc_score(test_targets[:, i], test_probs[:, i])),
            'f1': float(f1_score(test_targets[:, i], test_pred[:, i], zero_division=0)),
            'precision': float(precision_score(test_targets[:, i], test_pred[:, i], zero_division=0)),
            'recall': float(recall_score(test_targets[:, i], test_pred[:, i], zero_division=0)),
            'support': int(test_targets[:, i].sum()),
            'tier': triage_map[label]
        }
        for i, label in enumerate(LABELS)
    }
}

with open('D:/cxr-triage/notebooks/clahe_focal_320_final_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved to clahe_focal_320_final_results.json")